# Lab 02: Vector Search & Retrieval

## Overview

In this lab you build a production-ready **Vector Search index** on top of the
`arxiv_chunks` Delta table created in Lab 01.  You will:

1. Provision a **Vector Search Endpoint** (the compute that serves queries).
2. Create a **Delta Sync index** with **Managed Embeddings** using
   `databricks-bge-large-en` — Databricks handles embedding generation and
   incremental sync automatically.
3. Query the index using **semantic**, **keyword (BM25)**, and **hybrid** search.
4. Apply a **reranker** model to boost result quality.

**Exam Domain:** Data Preparation — *14 % of exam*

Relevant skills: Delta Sync indexes, managed embeddings, query types,
reranker usage, and embedding-model trade-offs.

In [ ]:
%pip install databricks-sdk databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "genai_lab_guide"
SCHEMA = "default"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.arxiv_chunks"
VS_ENDPOINT = "genai_lab_guide_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.arxiv_chunks_index"

print(f"Source table: {spark.table(SOURCE_TABLE).count()} chunks")

## A. Create a Vector Search Endpoint

A **Vector Search Endpoint** is the compute resource that hosts and serves one or
more Vector Search indexes.  Endpoints are shared resources — a single endpoint can
serve many indexes — and they autoscale automatically.

Key points:
- You must create an endpoint before you can create any index.
- Endpoint names are workspace-scoped.
- The same endpoint can serve indexes backed by different embedding models.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

try:
    endpoint = vsc.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD",
    )
    print(f"Creating endpoint '{VS_ENDPOINT}' ...")
    vsc.wait_get_endpoint_ready(VS_ENDPOINT)
    print("Endpoint is ready.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists — skipping creation.")
    else:
        raise

print(vsc.get_endpoint(VS_ENDPOINT))

## B. Create a Delta Sync Vector Search Index

A **Delta Sync index** stays in sync with the source Delta table automatically.
By choosing **Managed Embeddings** you delegate embedding generation to Databricks
using `databricks-bge-large-en` — you do not need to call an embedding model
yourself.

Prerequisites (already satisfied from Lab 01):
1. Source table has a **unique primary key** column (`chunk_id`).
2. Source table has **Change Data Feed** (`delta.enableChangeDataFeed = true`) enabled.

`pipeline_type="TRIGGERED"` means the index refreshes on demand (cost-efficient
for a lab).  In production you would use `"CONTINUOUS"` for near-real-time updates.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        source_table_name=SOURCE_TABLE,
        index_name=VS_INDEX,
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name="databricks-bge-large-en",
        pipeline_type="TRIGGERED",
    )
    print(f"Creating index '{VS_INDEX}' ...")
    index.wait_until_ready()
    print("Index is ready.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{VS_INDEX}' already exists — skipping creation.")
        index = vsc.get_index(VS_ENDPOINT, VS_INDEX)
    else:
        raise

print(index.describe())

## C. Semantic Search

**Semantic search** finds documents whose *meaning* is similar to the query,
even when no exact words match.  Under the hood Databricks:

1. Embeds the query text using the same model as the index (`databricks-bge-large-en`).
2. Computes cosine similarity between the query embedding and every stored embedding.
3. Returns the top-*k* chunks sorted by similarity score.

Use semantic search when the user query is expressed in natural language and you
want conceptual matches rather than exact keyword hits.

In [ ]:
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

results = index.similarity_search(
    query_text="How does the attention mechanism work in transformers?",
    columns=["chunk_id", "chunk_text", "path"],
    num_results=5,
)

print("=== Semantic Search Results ===")
for row in results["result"]["data_array"]:
    chunk_id, chunk_text, path, score = row
    source = path.split("/")[-1] if path else "unknown"
    print(f"Score: {score:.4f} | Source: {source}")
    print(f"  {chunk_text[:200]}...")
    print()

## D. Keyword Search

**Keyword search** uses BM25 (Best Matching 25), a classic term-frequency ranking
algorithm.  It scores documents based on how often query terms appear relative to
the document length and the overall corpus frequency.

BM25 excels when:
- The query contains specific technical terms, acronyms, or proper nouns.
- Exact terminology matters (e.g., "BERT", "LoRA", "GPT-4").
- The concept is not well represented in the embedding space.

BM25 struggles with synonyms and paraphrasing — a query for "car" will not match
a document that only uses "automobile".

In [ ]:
results_kw = index.similarity_search(
    query_text="BERT masked language model pre-training",
    columns=["chunk_id", "chunk_text", "path"],
    num_results=5,
    query_type="keyword",
)

print("=== Keyword (BM25) Search Results ===")
for row in results_kw["result"]["data_array"]:
    chunk_id, chunk_text, path, score = row
    source = path.split("/")[-1] if path else "unknown"
    print(f"Score: {score:.4f} | Source: {source}")
    print(f"  {chunk_text[:200]}...")
    print()

## E. Hybrid Search

**Hybrid search** combines semantic (embedding) similarity with keyword (BM25)
matching using Reciprocal Rank Fusion (RRF).  RRF merges ranked lists from both
retrievers: a chunk that appears high in *both* lists gets a strong combined score.

When to use hybrid:
- General-purpose RAG pipelines where query types are unpredictable.
- Queries that mix natural language with technical terms.
- When you want the precision of keyword search and the recall of semantic search.

Hybrid is the recommended default for most production RAG applications.

In [ ]:
results_hybrid = index.similarity_search(
    query_text="Low-rank adaptation fine-tuning efficiency",
    columns=["chunk_id", "chunk_text", "path"],
    num_results=5,
    query_type="hybrid",
)

print("=== Hybrid Search Results ===")
for row in results_hybrid["result"]["data_array"]:
    chunk_id, chunk_text, path, score = row
    source = path.split("/")[-1] if path else "unknown"
    print(f"Score: {score:.4f} | Source: {source}")
    print(f"  {chunk_text[:200]}...")
    print()

## F. Comparing Embedding Models

Databricks hosts several BGE models out of the box.  Choosing the right one
involves a cost-vs-quality trade-off:

| Model | Size | Dimension | Throughput | Best for |
|-------|------|-----------|------------|----------|
| `databricks-bge-large-en` | 0.44 GB | 1 024 | Medium | Higher accuracy, complex queries |
| `databricks-bge-small-en` | 0.13 GB | 384 | High | Cost-sensitive, high-volume pipelines |

**Key trade-offs:**

- **Higher dimension** → richer representation → better semantic recall, but more
  storage and slower ANN search.
- **Larger model** → higher embedding quality, but more GPU memory and higher
  inference cost per token.
- Once an index is created with a given model, changing the model requires
  **rebuilding the index from scratch** — choose wisely upfront.
- For the exam: `bge-large` = better quality; `bge-small` = lower cost.

## G. Adding a Reranker

A **reranker** is a cross-encoder model that re-scores an initial set of
retrieved candidates.  Unlike the bi-encoder used for retrieval (which encodes
query and document independently), a reranker processes both together — giving it
much better relevance judgement.

Reranker workflow:

1. Retrieve *N* candidate chunks via hybrid search (e.g., top 20).
2. Pass each (query, chunk) pair through the reranker.
3. Re-sort by reranker score and return the top *k* (e.g., top 5).

Important nuances:
- A reranker **does not add new documents** — it only reorders what retrieval already found.
- It adds latency and cost, so use it when precision matters more than speed.
- Databricks provides `databricks-reranker-v1` as a managed endpoint.

In [ ]:
results_reranked = index.similarity_search(
    query_text="Low-rank adaptation fine-tuning efficiency",
    columns=["chunk_id", "chunk_text", "path"],
    num_results=5,
    query_type="hybrid",
    query_vector_search_config={
        "reranker": {
            "reranker_model_endpoint_name": "databricks-reranker-v1",
        }
    },
)

print("=== Hybrid Search with Reranker ===")
for row in results_reranked["result"]["data_array"]:
    chunk_id, chunk_text, path, score = row
    source = path.split("/")[-1] if path else "unknown"
    print(f"Score: {score:.4f} | Source: {source}")
    print(f"  {chunk_text[:200]}...")
    print()

## Key Takeaways

1. **Endpoint first, index second.** A Vector Search Endpoint must exist before you
   can create any index.  One endpoint serves many indexes.

2. **Delta Sync prerequisites.** The source Delta table must have a unique primary
   key and Change Data Feed enabled — both were set up in Lab 01.

3. **Managed Embeddings = zero embedding code.** Databricks auto-generates and
   stores embeddings when `embedding_source_column` and
   `embedding_model_endpoint_name` are specified.

4. **Semantic vs. Keyword vs. Hybrid.**  Use semantic for natural-language queries,
   keyword for exact terms, and hybrid (recommended default) for mixed workloads.
   Hybrid uses Reciprocal Rank Fusion under the hood.

5. **Rerankers improve precision, not recall.** They reorder existing candidates —
   they do not surface new documents.  Apply them when the top-5 quality matters
   more than raw throughput.

6. **Embedding model trade-off.** `bge-large` (1 024 dim) delivers better quality;
   `bge-small` (384 dim) is faster and cheaper.  Changing the model requires
   rebuilding the index.

---

Continue to [`workbook.md`](workbook.md) for the architecture diagram, exam
question patterns, and a cost breakdown.